# EDA — MS Brain MRI Lesion Segmentation dataset

Dataset: [orvile/multiple-sclerosis-brain-mri-lesion-segmentation](https://www.kaggle.com/datasets/orvile/multiple-sclerosis-brain-mri-lesion-segmentation) (60 MS patients, T1/T2/FLAIR + consensus lesion mask, NIfTI format).

Run `python src/data/download.py` first (requires a configured Kaggle API token — see the README) so that `data/raw/` is populated before running this notebook.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

from src.data.preprocessing import discover_patients, patient_id_from_dir, load_patient_volumes

RAW_DIR = PROJECT_ROOT / "data" / "raw"
patient_dirs = discover_patients(RAW_DIR)
print(f"Found {len(patient_dirs)} patient folders under {RAW_DIR}")
patient_dirs[:5]

## Inspect a single patient's volumes

In [ ]:
sample_dir = patient_dirs[0]
sample_id = patient_id_from_dir(sample_dir)
volumes = load_patient_volumes(sample_dir)

for name, vol in volumes.items():
    print(f"{name}: shape={vol.shape}, dtype={vol.dtype}, min={vol.min():.2f}, max={vol.max():.2f}")

## Visualize a central axial slice: T1 / T2 / FLAIR + lesion mask overlay

In [ ]:
from src.utils.viz import overlay_mask

z = volumes["t1"].shape[2] // 2
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, modality in zip(axes[:3], ["t1", "t2", "flair"]):
    ax.imshow(volumes[modality][:, :, z], cmap="gray")
    ax.set_title(modality.upper())
    ax.axis("off")

overlay_mask(axes[3], volumes["flair"][:, :, z], volumes["mask"][:, :, z])
axes[3].set_title("FLAIR + lesion mask")
fig.suptitle(f"Patient {sample_id}, slice {z}")
fig.tight_layout()

## Lesion load distribution across patients

Medical segmentation datasets are typically very imbalanced (most voxels are healthy tissue/background). This checks per-patient lesion burden and helps decide slice filtering strategy in preprocessing.

In [ ]:
records = []
for pdir in patient_dirs:
    pid = patient_id_from_dir(pdir)
    vols = load_patient_volumes(pdir)
    mask = vols["mask"]
    lesion_voxels = int(mask.sum())
    slices_with_lesion = int((mask.sum(axis=(0, 1)) > 0).sum())
    total_slices = mask.shape[2]
    records.append({
        "patient_id": pid,
        "lesion_voxels": lesion_voxels,
        "slices_with_lesion": slices_with_lesion,
        "total_slices": total_slices,
        "pct_slices_with_lesion": 100 * slices_with_lesion / total_slices,
    })

import pandas as pd
lesion_df = pd.DataFrame(records)
lesion_df.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(lesion_df["patient_id"], lesion_df["lesion_voxels"])
ax.set_xlabel("Patient")
ax.set_ylabel("Lesion voxel count")
ax.set_title("Lesion burden per patient")
plt.xticks(rotation=90)
fig.tight_layout()

## Clinical metadata

If the Kaggle download includes a metadata file (EDSS, demographics, etc.), load it here — the exact filename/format wasn't confirmed before download, so inspect `data/raw/` directly (e.g. `list((PROJECT_ROOT / 'data' / 'raw').glob('*.csv'))` or `*.xlsx`) and adjust the loading code below accordingly.

In [ ]:
metadata_candidates = list(RAW_DIR.glob("*.csv")) + list(RAW_DIR.glob("*.xlsx"))
print("Candidate metadata files:", metadata_candidates)